# Round 4 — Rule-specific lexical evidence

One bounded feature-representation experiment. Keep the classifier fixed and reuse ten saved control fits. Twelve new CPU fits. No model downloads, new neural inference, ensemble or automatic submission. The 881-comment development cohort has been repeatedly inspected; it is not an independent holdout or Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/evidence_features.json").is_file())
from scripts.run_evidence_features import verify_prior, bounded_compute, figures, write_dashboard
config = json.loads((ROOT / "configs/evidence_features.json").read_text())
print("Project:", ROOT)
print("Primary:", config["primary"])
print("New CPU fits:", config["new_fits"], "| Prior fits reused:", config["cached_control_fits"])

Project: /home/sagemaker-user/projects/jigsaw-rule-classifier
Primary: rule_both
New CPU fits: 12 | Prior fits reused: 10


## 1. What led here

Round 3 policy-conditioned lexical features improved the actor anchor by +0.00842 mean AUC, but its simultaneous interval crossed zero. Both policies improved; dense policy interactions did not help. Keep that decision unpromoted. This follow-up asks whether lexical relevance under a rule—not just corpus rarity—deserves explicit representation.

In [2]:
prior = verify_prior(ROOT, config)
previous = prior[4]
display(pd.DataFrame(previous["pooled_metrics"]))
print("Previous decision:", previous["decision"])
print("Old source and candidate checkpoints verified; no prior models refitted.")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,lexical_control,0.657508,0.641345,0.650247
1,add_behavior,0.679862,0.680777,0.681784
2,add_act_roles,0.686099,0.691055,0.690944
3,copy_lexical,0.684815,0.690370,0.690285
4,condition_lexical,0.694518,0.698787,0.698227
5,copy_behavior,0.685083,0.689758,0.689774
6,condition_behavior,0.683636,0.688308,0.688402
7,copy_both,0.683560,0.688946,0.688677
8,condition_both,0.692901,0.697225,0.696993


Previous decision: DO_NOT_PROMOTE_PRIMARY
Old source and candidate checkpoints verified; no prior models refitted.


## 2. Evidence weighting, without extra columns

Fit smoothed violating/permitted document-incidence ratios on eligible training only. Shrink local-rule evidence toward global evidence when the smaller unique training class is small. Weights are bounded to [1,4], and each word/character row keeps its original norm. Reweight only the added policy-specific lexical block; leave shared lexical and dense features unchanged.

Controls: global evidence, word-only, character-only, both, unshrunk local evidence, and a fixed within-family permutation of the learned weights. Equal dimensions, sparsity and row norms do not equalize the intended per-coordinate regularization effect. No invented negative labels or query-label fitting.

In [3]:
result = bounded_compute(ROOT)
print("Decision:", result["decision"])
print("New fits:", result["new_fits"])
print("Old fits reused:", result["reused_prior_controls"])
print("Control prediction parity:", result["control_design_parity"])
CHARTS = figures(result)

{"timestamp": "2026-09-12T00:03:25+00:00", "stage": "evidence_round4", "event": "started", "elapsed_seconds": 0.0, "stage_elapsed_seconds": 0.0, "total_elapsed_seconds": 0.0}


{"timestamp": "2026-09-12T00:03:34+00:00", "stage": "evidence_round4", "event": "all_control_designs_verified", "elapsed_seconds": 9.146, "stage_elapsed_seconds": 9.146, "total_elapsed_seconds": 9.146, "cached_controls": 10, "new_fits": 0}


{"timestamp": "2026-09-12T00:03:35+00:00", "stage": "evidence_round4", "event": "candidate_complete", "elapsed_seconds": 10.495, "stage_elapsed_seconds": 10.495, "total_elapsed_seconds": 10.495, "completed": 1, "total": 12, "fold": 0, "variant": "global_both", "new_fits": 1, "reused_fits": 0}


{"timestamp": "2026-09-12T00:03:37+00:00", "stage": "evidence_round4", "event": "candidate_complete", "elapsed_seconds": 11.837, "stage_elapsed_seconds": 11.837, "total_elapsed_seconds": 11.837, "completed": 2, "total": 12, "fold": 0, "variant": "rule_words", "new_fits": 2, "reused_fits": 0}


{"timestamp": "2026-09-12T00:03:38+00:00", "stage": "evidence_round4", "event": "candidate_complete", "elapsed_seconds": 13.141, "stage_elapsed_seconds": 13.141, "total_elapsed_seconds": 13.141, "completed": 3, "total": 12, "fold": 0, "variant": "rule_chars", "new_fits": 3, "reused_fits": 0}


...

{"timestamp": "2026-09-12T00:03:46+00:00", "stage": "evidence_round4", "event": "candidate_complete", "elapsed_seconds": 20.851, "stage_elapsed_seconds": 20.851, "total_elapsed_seconds": 20.851, "completed": 10, "total": 12, "fold": 1, "variant": "rule_both", "new_fits": 10, "reused_fits": 0}


{"timestamp": "2026-09-12T00:03:47+00:00", "stage": "evidence_round4", "event": "candidate_complete", "elapsed_seconds": 21.836, "stage_elapsed_seconds": 21.836, "total_elapsed_seconds": 21.836, "completed": 11, "total": 12, "fold": 1, "variant": "permuted_both", "new_fits": 11, "reused_fits": 0}


{"timestamp": "2026-09-12T00:03:48+00:00", "stage": "evidence_round4", "event": "candidate_complete", "elapsed_seconds": 22.817, "stage_elapsed_seconds": 22.817, "total_elapsed_seconds": 22.817, "completed": 12, "total": 12, "fold": 1, "variant": "unshrunk_both", "new_fits": 12, "reused_fits": 0}


{"timestamp": "2026-09-12T00:03:48+00:00", "stage": "evidence_round4", "event": "results_saved", "elapsed_seconds": 23.247, "stage_elapsed_seconds": 23.247, "total_elapsed_seconds": 23.247, "decision": "DO_NOT_PROMOTE_PRIMARY", "new_fits": 12}
{"timestamp": "2026-09-12T00:03:48+00:00", "stage": "evidence_round4", "event": "completed", "elapsed_seconds": 23.248, "stage_elapsed_seconds": 23.248, "total_elapsed_seconds": 23.248, "error_type": null}
RESULT: EVIDENCE_ROUND_COMPLETE DECISION: DO_NOT_PROMOTE_PRIMARY


Decision: DO_NOT_PROMOTE_PRIMARY
New fits: 12
Old fits reused: 10
Control prediction parity: True


## 3. Performance and uncertainty

Inspect both policy cohorts rather than only the mean. Intervals cover this round’s 12 planned contrasts, not all adaptive research choices or refitting variability.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss,queries
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527,234
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719,234
2,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,0.675784,0.231472,0.663380,234
3,0,"No Advertising: Spam, referral links, unsolici...",copy_lexical,0.673097,0.237143,0.684000,234
4,0,"No Advertising: Spam, referral links, unsolici...",condition_lexical,0.686604,0.231335,0.669720,234
5,0,"No Advertising: Spam, referral links, unsolici...",global_both,0.687500,0.231638,0.672520,234
6,0,"No Advertising: Spam, referral links, unsolici...",rule_words,0.690634,0.231305,0.668772,234
7,0,"No Advertising: Spam, referral links, unsolici...",rule_chars,0.685410,0.231414,0.670310,234
8,0,"No Advertising: Spam, referral links, unsolici...",rule_both,0.688694,0.231786,0.670448,234
9,0,"No Advertising: Spam, referral links, unsolici...",permuted_both,0.687052,0.231094,0.670799,234


## 4. Does the learned association actually matter?

The primary must beat both the previous policy anchor and the matched weight-permutation control. Word/character removals identify conditional family value. Local-versus-global and shrunk-versus-unshrunk are mechanism diagnostics, not a chance to replace the registered primary.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

,comparison,candidate,reference,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,global_both vs policy anchor,global_both,condition_lexical,0.006695,-0.002964,0.016354,500
1,rule_words vs policy anchor,rule_words,condition_lexical,0.005826,-0.003833,0.015485,500
2,rule_chars vs policy anchor,rule_chars,condition_lexical,0.003679,-0.005980,0.013338,500
3,rule_both vs policy anchor,rule_both,condition_lexical,0.007728,-0.001931,0.017386,500
4,permuted_both vs policy anchor,permuted_both,condition_lexical,0.000591,-0.009068,0.010250,500
5,unshrunk_both vs policy anchor,unshrunk_both,condition_lexical,0.007081,-0.002578,0.016739,500
6,Rule evidence versus global evidence,rule_both,global_both,0.001032,-0.008626,0.010691,500
7,Learned weights versus permuted weights,rule_both,permuted_both,0.007137,-0.002522,0.016796,500
8,Effect of shrinkage,rule_both,unshrunk_both,0.000647,-0.009012,0.010306,500
9,Ablation: character reweighting,rule_both,rule_words,0.001902,-0.007757,0.011560,500


## 5. Training evidence strength and scale checks

No raw token strings are displayed or exported. Quantiles describe fitted evidence weights, not importance. The shrinkage reliability count uses unique pairs rather than repeated training occurrences. Every candidate preserves its word/character row norms and the previous design width and sparsity.

In [6]:
display(pd.DataFrame(result["weight_profiles"]))
print("Norm checks:", len(result["norm_checks"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,policy,permitted_pairs,violating_pairs,local_evidence_fraction,fallback_global,family,fold,mode,columns,minimum,q25,median,q75,maximum,capped_columns
0,"no advertising: spam, referral links, unsolici...",379,250,0.796178,False,word,0,global,8404,1.001942,1.251973,1.759628,2.319814,4.000000,57
1,"no advertising: spam, referral links, unsolici...",379,250,0.796178,False,word,0,rule,8404,1.000575,1.182734,1.406655,2.140067,4.000000,51
2,"no advertising: spam, referral links, unsolici...",379,250,0.796178,False,word,0,unshrunk,8404,1.004782,1.246532,1.246532,2.345145,4.000000,54
3,no legal advice: do not offer or request legal...,422,589,0.868313,False,word,0,global,8404,1.001942,1.251973,1.759628,2.319814,4.000000,57
4,no legal advice: do not offer or request legal...,422,589,0.868313,False,word,0,rule,8404,1.000330,1.333970,1.478644,1.764642,3.710245,0
5,no legal advice: do not offer or request legal...,422,589,0.868313,False,word,0,unshrunk,8404,1.000156,1.374538,1.374538,1.724075,3.760957,0
6,"no advertising: spam, referral links, unsolici...",379,250,0.796178,False,char,0,global,20000,1.003699,1.306171,1.680864,2.265046,4.000000,231
7,"no advertising: spam, referral links, unsolici...",379,250,0.796178,False,char,0,rule,20000,1.000776,1.360371,1.679605,2.352334,4.000000,273
8,"no advertising: spam, referral links, unsolici...",379,250,0.796178,False,char,0,unshrunk,20000,1.000181,1.345682,1.752930,2.444295,4.000000,347
9,no legal advice: do not offer or request legal...,422,589,0.868313,False,char,0,global,20000,1.003699,1.306171,1.680864,2.265046,4.000000,231


Norm checks: 12


## 6. Stability and probability quality

High-weight feature overlap across the two training folds is descriptive; stable features are not necessarily useful. Ranking improvements can coexist with worse log loss or Brier score. No calibration claim is made.

In [7]:
display(pd.DataFrame(result["weight_stability"]))
display(pd.DataFrame(result["pooled_metrics"]))
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

,family,policy,highest_weight_decile_jaccard,fold0_vocabulary,fold1_vocabulary,interpretation
0,word,"no advertising: spam, referral links, unsolici...",0.481560,8404,5643,"descriptive weight stability, not predictive p..."
1,word,no legal advice: do not offer or request legal...,0.196596,8404,5643,"descriptive weight stability, not predictive p..."
2,char,"no advertising: spam, referral links, unsolici...",0.502065,20000,20000,"descriptive weight stability, not predictive p..."
3,char,no legal advice: do not offer or request legal...,0.225866,20000,20000,"descriptive weight stability, not predictive p..."


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,lexical_control,0.657508,0.641345,0.650247
1,add_behavior,0.679862,0.680777,0.681784
2,add_act_roles,0.686099,0.691055,0.690944
3,copy_lexical,0.684815,0.690370,0.690285
4,condition_lexical,0.694518,0.698787,0.698227
5,global_both,0.701214,0.709018,0.707620
6,rule_words,0.700344,0.705257,0.704909
7,rule_chars,0.698197,0.705358,0.704208
8,rule_both,0.702246,0.709424,0.708569
9,permuted_both,0.695109,0.699562,0.698897


## 7. Fixed decision and research boundary

Primary: `rule_both`. Require +0.003 macro AUC, positive simultaneous lower bounds and no per-policy decline versus both `condition_lexical` and `permuted_both`. Ranked-pooled AUC cannot decline against the policy anchor. A secondary winner does not replace the primary.

This is NB-inspired feature weighting with logistic regression, not NB-SVM or contrastive encoder training. [Wang and Manning (2012)](https://aclanthology.org/P12-2018/) motivates class-diagnostic weighting; [Daumé III (2007)](https://aclanthology.org/P07-1033/) motivates the existing shared/policy blocks. Historical generic NB sensitivity is distinct from this support-informed, norm-controlled test. Feature research remains open; the accepted Qwen model is not evaluated or changed here.

In [8]:
for requirement in result["primary_requirements"]:
    display(pd.DataFrame([requirement["contrast"]]))
    print("Reference:", requirement["reference"], "| Policy deltas:", requirement["per_policy_delta"])
print("Decision:", result["decision"])
for note in result["limitations"]:
    print(note)
print("Dashboard:", write_dashboard(ROOT, result))
print("Private checkpoints:", ROOT / "runs/evidence_features" / result["run_id"])

,comparison,candidate,reference,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,rule_both vs policy anchor,rule_both,condition_lexical,0.007728,-0.001931,0.017386,500


Reference: condition_lexical | Policy deltas: [0.002089552238805914, 0.013365687559930395]


,comparison,candidate,reference,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,Learned weights versus permuted weights,rule_both,permuted_both,0.007137,-0.002522,0.016796,500


Reference: permuted_both | Policy deltas: [0.0016417910447761308, 0.01263184673489759]
Decision: DO_NOT_PROMOTE_PRIMARY
Adaptive exploratory follow-up; not an independent holdout.
The rule-conditioned anchor was not promoted.
Training-only label evidence; query targets enter scoring only.
Supplied labels exist for the query rule: not zero-shot.
The permutation control fixes weight distributions, not all capacity.
Equal row norms do not equalize per-coordinate regularization.
Intervals cover 12 comparisons, not the full adaptive history.
No comparison to accepted-model predictions or new Kaggle score.
One permutation and one shrinkage strength; no tuning sweep.
Historical generic NB weighting is not evidence for this design.


Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/evidence_features/dashboard.html
Private checkpoints: /home/sagemaker-user/projects/jigsaw-rule-classifier/runs/evidence_features/e5628cf8367060756148
